### Explicación del Código

El código en `dividir_basedatos.ipynb` divide un archivo CSV en múltiples archivos más pequeños, asigna dinámicamente los archivos generados a personas y verifica que no se pierdan ni dupliquen filas en el proceso. Realiza las siguientes tareas:

1. **División del CSV**:
   - **Entrada**: Archivo CSV ubicado en `/home/sotavento/Documents/tejer_red/NER/ner_annotation/output/remove_duplicates_CSV_2025-04-01_00-06-10/filtered_output.csv`.
   - **Columna a procesar**: `descripcion_desaparicion`.
   - Divide el archivo en 8 partes (o según se configure) y guarda los archivos en una carpeta de salida única generada con un timestamp.

2. **Asignación de Archivos**:
   - Genera un archivo `assignment_log.csv` en la carpeta de salida, asignando cada archivo generado a una persona (e.g., `Person 1`, `Person 2`).

3. **Verificación de Integridad**:
   - Comprueba que no se pierdan ni dupliquen filas durante la división.
   - Genera un archivo de log `verification_log.txt` con los resultados de la verificación.

4. **Carpetas y Archivos Relevantes**:
   - **Entrada**: Archivo CSV especificado en `input_csv`.
   - **Salida**:
     - Archivos divididos: `output/output_<timestamp>/part_*.txt`.
     - Log de asignaciones: `output/output_<timestamp>/assignment_log.csv`.
     - Log de verificación: `output/output_<timestamp>/verification_log.txt`.

In [ ]:
# Instalar dependencias necesarias
!pip install pandas

In [ ]:
import pandas as pd
import os
from datetime import datetime
import re

def create_output_folder(base_output_folder):
    """
    Creates a unique output folder based on the current date and time.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_folder = os.path.join(base_output_folder, f"output_{timestamp}")
    os.makedirs(output_folder, exist_ok=True)
    return output_folder

def read_csv_file(input_csv):
    """
    Reads a CSV file without limiting the number of rows.
    """
    try:
        df = pd.read_csv(input_csv)
        return df
    except Exception as e:
        print(f"Error reading the CSV file: {e}")
        return None

def split_csv_random(input_csv, column, num_files=None, rows_per_file=None, max_rows=None, base_output_folder="output"):
    """
    Splits a CSV file into multiple smaller files by randomly shuffling rows across the entire dataset.
    Randomization is applied before limiting the number of rows.
    """
    # Create a unique output folder based on the current date and time
    output_folder = create_output_folder(base_output_folder)

    # Read the CSV file
    df = read_csv_file(input_csv)
    if df is None:
        return

    # Check if the column exists
    if column not in df.columns:
        print(f"The column '{column}' does not exist in the CSV file.")
        return

    # Randomly shuffle rows across the entire dataset
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle rows with a fixed random seed

    # Apply the max_rows limit after shuffling
    if max_rows:
        df = df.head(max_rows)

    # Log the shuffled and limited data
    print("Shuffled and limited data (first 5 rows):")
    print(df.head())

    # Extract the desired column
    column_data = df[[column]]

    # Check for missing values
    missing_rows = column_data[column].isna().sum()
    print(f"Number of missing rows in column '{column}': {missing_rows}")

    # Drop rows with missing values
    column_data = column_data.dropna()

    # Split the data
    total_rows = len(column_data)
    if num_files:
        rows_per_file = total_rows // num_files + (total_rows % num_files > 0)
    elif not rows_per_file:
        print("You must specify either 'num_files' or 'rows_per_file'.")
        return

    # Save the split files as .txt
    for i in range(0, total_rows, rows_per_file):
        part = column_data.iloc[i:i + rows_per_file]
        output_file = os.path.join(output_folder, f"part_{i // rows_per_file + 1}.txt")
        part.to_csv(output_file, index=False, header=False)  # Save as .txt without index or header
        print(f"File created: {output_file} with {len(part)} rows.")

    print(f"Splitting completed. Files saved in: {output_folder}")
    return output_folder

def log_assignments(output_folder):
    """
    Dynamically logs who was assigned each file based on the number of generated files.
    
    :param output_folder: Folder where the log file will be saved.
    """
    # Get all split files in the output folder
    split_files = [os.path.join(output_folder, f) for f in os.listdir(output_folder) if f.startswith("part_") and f.endswith(".txt")]

    # Sort the split files using natural sorting
    split_files.sort(key=natural_sort_key)  # Sort files naturally (e.g., part_1.txt, part_2.txt, ..., part_10.txt)

    # Create a dynamic assignment for each file
    log_file = os.path.join(output_folder, "assignment_log.csv")
    with open(log_file, "w") as f:  # Use "w" to overwrite the log file each time
        for i, file in enumerate(split_files):
            f.write(f"{file},Person {i + 1}\n")
    print(f"Log updated: {log_file}")

def natural_sort_key(s):
    """
    Key function for natural sorting of filenames (e.g., part_1.txt, part_2.txt).
    """
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def verify_split(input_csv, column, max_rows, output_folder):
    """
    Verifies that the split process did not lose or duplicate any rows.
    Logs detailed information about the verification process.
    """
    print("Starting verification process...")
    log_file = os.path.join(output_folder, "verification_log.txt")
    with open(log_file, "w") as log:
        log.write(f"Verification process started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        log.write(f"Input CSV: {input_csv}\n")
        log.write(f"Column to verify: {column}\n")
        log.write(f"Max rows to process: {max_rows}\n")
        log.write(f"Output folder: {output_folder}\n\n")

        # Read the original CSV file
        df = read_csv_file(input_csv)
        if df is None:
            log.write("Error reading the original CSV file.\n")
            return

        # Randomly shuffle the original data with the same seed
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)

        # Apply the max_rows limit after shuffling
        if max_rows:
            df = df.head(max_rows)

        # Extract the original column data
        original_data = df[[column]].dropna().reset_index(drop=True)
        log.write(f"Original column data extracted. Total rows: {len(original_data)}\n")

        # Combine all split files
        split_files = [os.path.join(output_folder, f) for f in os.listdir(output_folder) if f.startswith("part_") and f.endswith(".txt")]
        split_files.sort(key=natural_sort_key)  # Sort files naturally
        log.write(f"Found {len(split_files)} split files in the output folder.\n")

        combined_data = pd.DataFrame()
        for file in split_files:
            part = pd.read_csv(file, header=None, names=[column])
            combined_data = pd.concat([combined_data, part], ignore_index=True)

        # Verify row count
        if len(original_data) != len(combined_data):
            log.write(f"Verification failed: Row count mismatch.\n")
            print(f"Verification failed: Row count mismatch. Check {log_file} for details.")
            return

        # Verify data integrity
        if not original_data.equals(combined_data):
            log.write("Verification failed: Data mismatch or duplicates found.\n")
            print(f"Verification failed: Data mismatch. Check {log_file} for details.")
            return

        log.write("Verification successful: No rows lost or duplicated.\n")
        print("Verification successful: No rows lost or duplicated.")

# Example usage
if __name__ == "__main__":
    input_csv = "/home/sotavento/Documents/tejer_red/NER/ner_annotation/output/remove_duplicates_CSV_2025-04-01_00-06-10/filtered_output.csv"
    column = "descripcion_desaparicion"
    num_files = 8
    rows_per_file = None
    max_rows = None

    base_output_folder = "output"

    output_folder = split_csv_random(input_csv, column, num_files=num_files, rows_per_file=rows_per_file, max_rows=max_rows, base_output_folder=base_output_folder)
    if output_folder:
        log_assignments(output_folder)
        verify_split(input_csv, column, max_rows, output_folder)